# ENA and ONA with RS.data — pyENA + qeviz

This notebook mirrors the R vignette `rena-demo.Rmd`.
It fits a standard **ENA** (undirected) model and a **mean-rotation ENA**
model from the `RS.data` dataset using **pyENA**, then renders both as
interactive qeviz plots.

**Codes:** Data · Technical Constraints · Performance Parameters ·  
Client and Consultant Requests · Design Reasoning · Collaboration  
**Groups:** FirstGame · SecondGame  
**Units:** Condition × UserName (composite key)  
**Conversations:** GroupName

## Setup

Install the required packages if not already present:

```bash
# qeviz (from the qeviz repo root)
pip install -e ../

# pyENA (from the rENA repo)
pip install -e /path/to/rENA/python
```

In [ ]:
import pandas as pd
import qeviz
from pyena import ENA
from pyena.rotations import mean_rotation

## Load RS.data

`RS.data` is included at `../data/RS.data.csv` relative to this notebook.
It can also be exported from R:
```r
library(tma)
data(RS.data)
write.csv(RS.data, "RS.data.csv", row.names = FALSE)
```

In [ ]:
df = pd.read_csv("../data/RS.data.csv")
print(df.shape)
df.head(3)

## Prepare data

pyENA takes a single `units` column. Mirror the R convention of
`units_by = c("Condition", "UserName")` by creating a composite key.

In [ ]:
CODES = [
    "Data",
    "Technical.Constraints",
    "Performance.Parameters",
    "Client.and.Consultant.Requests",
    "Design.Reasoning",
    "Collaboration",
]

df["unit_key"] = df["Condition"] + "::" + df["UserName"]

print(f"{df['unit_key'].nunique()} units across "
      f"{df['GroupName'].nunique()} conversations")
print("Conditions:", df["Condition"].unique().tolist())

## Accumulate and fit — SVD rotation (standard ENA)

The default SVD rotation maximises variance across all units.
This mirrors `rENA::model(result_ena)` without a specific rotation.

In [ ]:
model_svd = ENA().fit(
    df,
    units         = "unit_key",
    conversations = "GroupName",
    codes         = CODES,
    window_size   = 8,
)

print(f"Units:       {len(model_svd.units_)}")
print(f"Codes:       {model_svd.codes_}")
print(f"Connections: {len(model_svd.connection_names_)}")
print(f"Points shape: {model_svd.points_.shape}")

## Visualise — standard ENA

`qeviz.from_pyena()` converts the fitted model to a `QEPlot`.
Chain `.edges()` and `.points()` to configure the view;
the object auto-displays in Jupyter via `_repr_html_`.

In [ ]:
qe_svd = qeviz.from_pyena(model_svd, group_col="Condition", width=720, height=580)

groups = qe_svd.groups()
print("Groups:", groups)

# Overlay: both group means + each group's network
qe_svd.edges(groups[0], also=groups[1]).points()

## Fit — mean rotation

Mean rotation places the mean difference between the two groups on SVD1,
maximising separation.  This is the rotation most commonly used when
comparing two conditions — equivalent to `rENA::ena.rotate.by.mean`.

In [ ]:
# Build group masks from the unit-level metadata produced by accumulation.
# We need to accumulate first so we have accum_.meta, then pass the
# same accumulation to fit() with the rotation.
from pyena import accumulate

accum = accumulate(
    df,
    units         = "unit_key",
    conversations = "GroupName",
    codes         = CODES,
    window_size   = 8,
)

g1_mask = accum.meta["Condition"] == "FirstGame"
g2_mask = accum.meta["Condition"] == "SecondGame"

model_mr = ENA().fit(
    accum,
    rotation = mean_rotation(g1_mask.values, g2_mask.values),
)

print("Mean-rotation model fitted.")

## Visualise — mean rotation

In [ ]:
qe_mr = qeviz.from_pyena(model_mr, group_col="Condition", width=720, height=580)

qe_mr.edges(groups[0], also=groups[1]).points()

## Export to standalone HTML

Both plots are self-contained — the qeviz bundle is inlined, so the files
open in any browser without a server.

In [ ]:
(
    qe_svd.edges(groups[0], also=groups[1]).points()
         .export_html("ena_svd.html")
)
(
    qe_mr.edges(groups[0], also=groups[1]).points()
        .export_html("ena_mean_rotation.html")
)
print("Wrote ena_svd.html and ena_mean_rotation.html")

## Individual group views

Pass a single `group` option to see one condition's network in isolation.

In [ ]:
for g in groups:
    print(f"\n── {g} ──")
    qe_mr.edges(g).points().show()

## Subtraction view

Pass `compare` to show the difference network (FirstGame − SecondGame).

In [ ]:
qe_mr.edges(groups[0], compare=groups[1])

## Individual unit network

Use `.units()` to list available unit IDs, then `.edges(unit=...)` to
plot one unit's personal connection network alongside the group means.

In [ ]:
# List available unit IDs
print(qe_mr.units()[:6], "...")

# Plot one unit's network with all group means in the background
unit_id = qe_mr.units()[0]
qe_mr.group().points().edges(unit=unit_id)